# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) and is accessible via the following schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We'll start by reading the Croissant schema metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Display high-level metadata
print(f"Title: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.date_published}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review the available record sets in the dataset, including their `@id` values, names, and the fields/columns they contain. We'll print them to help you select which record sets or fields to analyze.

In [ ]:
# List all RecordSets available in the dataset
print("Record Sets in dataset:")
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id}")
    print(f"  name: {rs.name}")
    if rs.description:
        print(f"  description: {rs.description}")
    print(f"  Fields/Columns:")
    for field in rs.fields:
        print(f"    - @id: {field.id} | name: {field.name} | type: {field.data_type}")
    print()

## 2a. Preview Data in a Record Set
As an example, let's print several records from the first major (primary) RecordSet. Adjust `record_set_id` as needed for the set you wish to inspect.


In [ ]:
# Pick the main RecordSet for proteins data (use @id exactly as printed above, example below)
# If unsure, list dataset.metadata.record_sets to see full list
record_set_id = None
for rs in dataset.metadata.record_sets:
    if 'protein' in (rs.name or '').lower() or True:
        record_set_id = rs.id
        break

print(f"Previewing some records from RecordSet @id: {record_set_id}")

count = 0
for rec in dataset.records(record_set=record_set_id):
    print(rec)
    count += 1
    if count >= 3:  # Only print a few records
        break

## 3. Data Extraction
Let's load all data from selected record sets into pandas DataFrames. Use the `@id` values from above for precise references. You can adjust `record_set_ids` to include any subsets of interest.


In [ ]:
# List all record sets @ids you want to analyze:
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns in {rs_id}: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records found in {rs_id}.")

# Choose the main RecordSet (should match 'proteins' or similar)
main_record_set_id = record_set_ids[0]  # Adjust if needed
main_df = dataframes[main_record_set_id]
print(f"\nMain DataFrame columns: {main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
We'll filter, normalize, and group numeric columns in the main DataFrame. Use field `@id`s from earlier for accurate column reference.

For demonstration, let's choose a numeric field/column, e.g. `coverage` (often present in protein mass spectrometry data, adjust as needed to your schema).

In [ ]:
# Pick candidate fields (@id and column name) for analysis
import numpy as np

# Find the first numeric field in the main record set
numeric_field_id = None
for field in dataset.metadata.record_set(main_record_set_id).fields:
    if field.data_type.lower() in {'number', 'integer', 'float'}:
        numeric_field_id = field.id
        print(f"Chosen numeric field for analysis: @id={numeric_field_id}, name={field.name}")
        break

if numeric_field_id is None:
    # Fallback to a likely candidate - try to guess from columns
    for col in main_df.columns:
        if 'coverage' in col.lower() or 'mw' in col.lower() or 'abundance' in col.lower():
            numeric_field_id = col
            print(f"Guessed numeric field: {numeric_field_id}")
            break

# Filter records with numeric_field > threshold (pick threshold)
if numeric_field_id in main_df.columns:
    # Drop rows with missing values to safely convert
    col = numeric_field_id
    main_df = main_df.dropna(subset=[col])
    main_df[col] = pd.to_numeric(main_df[col], errors='coerce')
    threshold = main_df[col].mean() if not np.isnan(main_df[col].mean()) else 10
    filtered_df = main_df[main_df[col] > threshold]
    print(f"Filtered records with {col} > {threshold:.2f} (mean): {len(filtered_df)} records")
    print(filtered_df.head())

    # Normalize numeric_field
    filtered_df[f"{col}_normalized"] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
    print(f"\nNormalized {col}:")
    print(filtered_df[[col, f"{col}_normalized"]].head())

    # Pick a group_field for grouping (e.g., description or accession)
    group_field_candidates = [c for c in main_df.columns if 'description' in c.lower() or 'accession' in c.lower()]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped by {group_field}:\n", grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and a relationship with another numeric field (if available). 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Look for a second numeric column for scatterplot
    other_numeric_cols = [c for c in main_df.columns if c != numeric_field_id and pd.api.types.is_numeric_dtype(main_df[c])]
    if other_numeric_cols:
        scatter_col = other_numeric_cols[0]
        plt.figure(figsize=(7,5))
        sns.scatterplot(data=main_df, x=numeric_field_id, y=scatter_col)
        plt.title(f'{numeric_field_id} vs {scatter_col}')
        plt.xlabel(numeric_field_id)
        plt.ylabel(scatter_col)
        plt.show()
    else:
        print("Only one numeric column found; skipping scatter plot.")

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR<sup>2</sup> mass spectrometry dataset using `mlcroissant`. We've reviewed the available record sets, loaded the primary protein data, performed basic filtering and normalization, grouped data for summary, and visualized key variables. 

You can extend this notebook by exploring additional fields, combining record sets, or running more advanced statistical analysis.